# 🏦 Fine-Tuning TinyLlama for Financial Research Q&A
### Using QLoRA on Google Colab (Free T4 GPU)

---

## 📌 What is this notebook doing? (Plain English)

We are teaching a pre-trained AI model new things about finance.

Here's the simple 3-step story:
1. **Start** with TinyLlama — a small AI that already knows a lot (it read the internet)
2. **Extra training** — we show it 2,000 finance Q&A examples so it gets better at finance topics
3. **Measure** — we compare how it answers finance questions before vs after the extra training

---

## 📚 Key Terms (Super Simple)

| Word | What it actually means |
|---|---|
| **Fine-tuning** | Extra training on a specific topic (like a doctor doing a medical specialization) |
| **QLoRA** | A smart trick to do fine-tuning without needing a expensive GPU |
| **Quantization** | Shrinking the model's numbers from 32-bit to 4-bit to save memory |
| **LoRA** | Adding tiny trainable layers to the model instead of changing everything |
| **Perplexity** | How 'confused' the model is. **Lower = better** |
| **Parameters** | The 1.1 billion numbers that make up TinyLlama's 'brain' |

---

## ⚠️ Before running: Enable GPU
> **Runtime → Change runtime type → T4 GPU → Save**

Without a GPU, training will take 10x longer and likely time out.

In [ ]:
# =============================================================
# CELL 1: Check that GPU is available
# =============================================================
# A GPU (Graphics Processing Unit) is specialized hardware that makes
# AI training 10-100x faster than a regular CPU.
# We need to confirm Colab gave us one.

import subprocess
result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
if result.returncode == 0:
    print(result.stdout)
else:
    print('ERROR: No GPU detected!')
    print('Please go to Runtime → Change runtime type → T4 GPU')

import torch
print(f'PyTorch version: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU Name: {torch.cuda.get_device_name(0)}')
    total_mem = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f'GPU Memory: {total_mem:.1f} GB')
    print('\n✅ You are good to go!')

## Step 0: Install the Tools

Before we can do anything, we need to install the Python libraries (tools) we'll use.

Think of libraries like apps on your phone:

| Library | What it does (simple version) |
|---|---|
| `transformers` | The main library by HuggingFace. Lets us download and use AI models. |
| `peft` | Implements LoRA (the memory-efficient fine-tuning trick) |
| `trl` | Training tools specifically for language models |
| `bitsandbytes` | Handles 4-bit quantization (making the model smaller) |
| `accelerate` | Makes PyTorch training faster and handles GPU setup |
| `datasets` | Downloads and manages datasets from HuggingFace |
| `evaluate` | Tools for measuring model performance |

In [ ]:
# =============================================================
# CELL 2: Install required libraries
# =============================================================
# The exclamation mark (!) means 'run this as a terminal command'
# -q means 'quiet' (less output)
# This takes 2-3 minutes. You'll see a lot of text scroll by -- that's normal.

!pip install -q \
    transformers==4.40.0 \
    peft==0.10.0 \
    trl==0.8.6 \
    bitsandbytes==0.43.1 \
    accelerate==0.29.3 \
    datasets \
    evaluate

print('\n✅ All libraries installed successfully!')

In [ ]:
# =============================================================
# CELL 3: Import everything we need
# =============================================================
# 'import' means 'load this library into memory so we can use it'

import torch
import math
from datasets import load_dataset
from transformers import (
    AutoModelForCausalLM,   # Loads the AI model
    AutoTokenizer,           # Loads the tokenizer (converts text <-> numbers)
    BitsAndBytesConfig,      # Configuration for 4-bit quantization
    TrainingArguments,       # Settings for the training process
)
from peft import (
    LoraConfig,                       # Configuration for LoRA
    TaskType,                         # Type of task (we're doing text generation)
    get_peft_model,                   # Applies LoRA to a model
    prepare_model_for_kbit_training,  # Prepares model for 4-bit training
)
from trl import SFTTrainer            # Supervised Fine-Tuning Trainer

print('✅ All imports successful!')
print(f'Using device: {"cuda (GPU) ⚡" if torch.cuda.is_available() else "cpu (slow!)"}')

## Step 1: Load the Dataset

### What is a dataset?
A dataset is just a big collection of examples used to train the model.

### Which dataset are we using?
We're using **finance-alpaca** from HuggingFace. It contains ~68,000 finance Q&A pairs.

Each example looks like:
```
Instruction: What is a stock?
Output: A stock represents ownership in a company...
```

We'll use **2,000 examples** for training and **200** for validation (checking quality while training).
Using all 68k would take too long on a free GPU.

In [ ]:
# =============================================================
# CELL 4: Load and explore the dataset
# =============================================================

print('Downloading the finance-alpaca dataset from HuggingFace...')
print('(This is ~68k finance Q&A pairs collected from the internet)')
print()

dataset = load_dataset('gbharti/finance-alpaca')

print(f'✅ Dataset loaded!')
print(f'Total examples available: {len(dataset["train"]):,}')
print()
print('=== Sample from the dataset ===')
example = dataset['train'][42]  # Pick example #42
print(f'Instruction: {example["instruction"]}')
print(f'Input (extra context): {example["input"] if example["input"] else "(none)"}')
print(f'Output (first 300 chars): {example["output"][:300]}...')

In [ ]:
# =============================================================
# CELL 5: Select a subset of data for training
# =============================================================
# We can't use all 68k examples (would take hours on free GPU)
# 2000 training + 200 validation is a good balance for speed vs. quality

TRAIN_SIZE = 2000
VAL_SIZE   = 200

# Shuffle so we get a diverse mix, not just the first 2000 examples
full_data = dataset['train'].shuffle(seed=42)  # seed=42 makes it reproducible

train_raw = full_data.select(range(TRAIN_SIZE))
val_raw   = full_data.select(range(TRAIN_SIZE, TRAIN_SIZE + VAL_SIZE))

print(f'Training examples: {len(train_raw)}')
print(f'Validation examples: {len(val_raw)}')
print()
print('Why validation? During training, we regularly check how the model')
print('performs on data it has NEVER seen. This tells us if it is actually')
print('learning general finance concepts, not just memorizing training examples.')

## Step 2: Format the Data

AI models don't just take raw text. They expect text in a **specific format** (like a script).

TinyLlama was trained with this chat format:
```
<|system|>
You are a helpful assistant.
</s>
<|user|>
What is inflation?
</s>
<|assistant|>
Inflation is the rate at which prices rise...
</s>
```

We need to convert every dataset example into this format before training. Think of it like filling out a standardized form.

In [ ]:
# =============================================================
# CELL 6: Format data into TinyLlama's chat format
# =============================================================

SYSTEM_PROMPT = (
    'You are a helpful financial research assistant. '
    'Answer questions about finance, investing, and economics '
    'accurately and clearly.'
)

def format_prompt(example):
    """
    This function takes one dataset example (a dict with 'instruction',
    'input', 'output') and converts it to TinyLlama's chat format.
    """
    instruction = example['instruction']
    input_ctx   = example.get('input', '') or ''   # sometimes empty
    output      = example['output']

    # If there's extra context, add it to the question
    if input_ctx.strip():
        user_message = f'{instruction}\n\nContext: {input_ctx}'
    else:
        user_message = instruction

    # Build the formatted text using TinyLlama's chat template
    text = (
        f'<|system|>\n{SYSTEM_PROMPT}\n</s>\n'
        f'<|user|>\n{user_message}\n</s>\n'
        f'<|assistant|>\n{output}\n</s>'
    )

    return {'text': text}

# Apply formatting to every single example in both datasets
train_data = train_raw.map(format_prompt, remove_columns=train_raw.column_names)
val_data   = val_raw.map(format_prompt, remove_columns=val_raw.column_names)

print('✅ Data formatted!')
print()
print('=== Formatted example (first 600 chars) ===')
print(train_data[0]['text'][:600])
print('...')

## Step 3: Load TinyLlama (The Base Model)

### What is TinyLlama?
- A 1.1 billion parameter language model
- Open-source (free to use and fine-tune)
- It was trained on 3 trillion tokens of text from the internet
- Think of it as a general-purpose student who has read everything

### What is 4-bit quantization?
Normally, each number in the model uses 32 bits of memory.
- 1.1B parameters × 32 bits = **~4.4 GB** just for the numbers
- 4-bit quantization: 1.1B parameters × 4 bits = **~0.55 GB**

We get almost the same quality at ~8x less memory. This is how we run it on a free GPU.

In [ ]:
# =============================================================
# CELL 7: Load TinyLlama in 4-bit quantized mode
# =============================================================

MODEL_NAME = 'TinyLlama/TinyLlama-1.1B-Chat-v1.0'

print('Configuring 4-bit quantization...')
# This config tells the model loader to compress weights to 4-bit
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,              # Load weights in 4-bit instead of 32-bit
    bnb_4bit_use_double_quant=True, # Apply quantization twice for better compression
    bnb_4bit_quant_type='nf4',      # NF4 = Normal Float 4, best quality 4-bit format
    bnb_4bit_compute_dtype=torch.bfloat16  # Use bfloat16 for the actual math operations
)

print('Loading tokenizer...')
# Tokenizer: converts text <-> numbers (tokens)
# E.g. 'inflation' might become token [12453]
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
tokenizer.pad_token = tokenizer.eos_token   # Use end-of-sentence token for padding
tokenizer.padding_side = 'right'            # Pad on the right side

print('Loading model... (this downloads ~600MB, may take 2-3 minutes)')
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map='auto'               # Automatically puts model on GPU
)
model.config.use_cache = False      # Disable cache for training (saves memory)

print()
print('✅ TinyLlama loaded!')
if torch.cuda.is_available():
    mem_used = torch.cuda.memory_allocated() / 1e9
    print(f'GPU memory used so far: {mem_used:.2f} GB')

## Step 4: Test the Model BEFORE Training (Baseline)

This is very important for the project!

We need to capture how the model performs **before** any fine-tuning, so we can show a genuine before/after comparison.

We measure two things:
1. **Qualitative** — Does it give good, relevant finance answers? (we read and judge)
2. **Perplexity** — A number measuring how 'confused' it is by finance text (lower = better)

In [ ]:
# =============================================================
# CELL 8: Define a function to generate responses
# =============================================================
# This function works both BEFORE and AFTER fine-tuning
# We'll call it twice to compare results

def generate_response(question, max_new_tokens=250):
    """
    Ask the model a question and get its answer.

    Parameters:
        question (str): The finance question to ask
        max_new_tokens (int): Maximum length of the answer

    Returns:
        str: The model's answer
    """
    # Format the question exactly like the training data format
    prompt = (
        f'<|system|>\n{SYSTEM_PROMPT}\n</s>\n'
        f'<|user|>\n{question}\n</s>\n'
        f'<|assistant|>\n'
    )

    # Convert text to tokens (numbers the model understands)
    inputs = tokenizer(prompt, return_tensors='pt').to(model.device)

    # Generate the response
    # torch.no_grad() tells PyTorch we are NOT training (saves memory)
    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=0.7,              # 0 = deterministic, 1 = very random. 0.7 is a good balance.
            do_sample=True,               # Use sampling (not greedy decoding)
            pad_token_id=tokenizer.eos_token_id,
            repetition_penalty=1.1        # Penalizes repeating the same phrases
        )

    # Decode: convert tokens back to text
    # We skip the input tokens and only decode the newly generated part
    new_tokens = output_ids[0][inputs['input_ids'].shape[1]:]
    response = tokenizer.decode(new_tokens, skip_special_tokens=True)
    return response.strip()

print('✅ generate_response() function defined!')
print('We will use this to test the model before and after training.')

In [ ]:
# =============================================================
# CELL 9: Define a function to calculate perplexity
# =============================================================
# Perplexity measures how well the model predicts finance text.
#
# Intuition: Give the model a finance sentence. Ask it to predict
# each next word. If it guesses right often, perplexity is low.
# If it's constantly surprised (wrong guesses), perplexity is high.
#
# Formula: perplexity = e^(average_loss)
# Lower perplexity = model understands this domain better

def calculate_perplexity(data, sample_size=100):
    """
    Calculate perplexity of the model on a given dataset.

    Parameters:
        data: Dataset with a 'text' column
        sample_size: How many examples to evaluate on

    Returns:
        float: Perplexity score (lower is better)
    """
    model.eval()  # Put model in evaluation mode (no dropout, no gradients)
    total_loss  = 0.0
    valid_count = 0

    texts = data['text'][:sample_size]

    for text in texts:
        # Tokenize (convert text to numbers)
        inputs = tokenizer(
            text,
            return_tensors='pt',
            max_length=512,
            truncation=True   # Cut off anything longer than 512 tokens
        )
        inputs = {k: v.to(model.device) for k, v in inputs.items()}

        # Get the model's loss on this text
        # When labels = input_ids, the model tries to predict each token
        # Loss tells us how wrong it was on average
        with torch.no_grad():
            outputs = model(**inputs, labels=inputs['input_ids'])
            loss = outputs.loss.item()

        # Skip any NaN or infinite values (shouldn't happen but just in case)
        if not (math.isnan(loss) or math.isinf(loss)):
            total_loss  += loss
            valid_count += 1

    if valid_count == 0:
        return float('inf')

    avg_loss   = total_loss / valid_count
    perplexity = math.exp(avg_loss)
    return round(perplexity, 2)

print('✅ calculate_perplexity() function defined!')

In [ ]:
# =============================================================
# CELL 10: Test the model BEFORE fine-tuning
# =============================================================
# These 5 questions will be our benchmark.
# We'll ask the exact same questions after training to compare.

TEST_QUESTIONS = [
    'What is a price-to-earnings (P/E) ratio and why does it matter to investors?',
    'Explain what portfolio diversification means and why it reduces investment risk.',
    'What is quantitative easing and how does it affect inflation and markets?',
    'What is the difference between a bull market and a bear market?',
    'What is compound interest and how does it grow wealth over time?',
]

print('=' * 65)
print('BASELINE: Model responses BEFORE fine-tuning')
print('=' * 65)
print()

before_responses = []
for i, question in enumerate(TEST_QUESTIONS, 1):
    print(f'Q{i}: {question}')
    print('-' * 50)
    response = generate_response(question)
    print(f'{response}')
    print()
    before_responses.append(response)

print('\nSave these responses! We will compare them after training.')

In [ ]:
# =============================================================
# CELL 11: Calculate baseline perplexity
# =============================================================
# This gives us a quantitative (number-based) baseline
# before any fine-tuning happens.

print('Calculating baseline perplexity on 100 finance examples...')
print('(This takes ~2 minutes - the model is evaluating each example)')
print()

baseline_perplexity = calculate_perplexity(val_data, sample_size=100)

print(f'✅ Baseline Perplexity: {baseline_perplexity}')
print()
print('What this means:')
print('  - This is the model\'s starting point BEFORE seeing any finance training data.')
print('  - After fine-tuning, this number should decrease.')
print('  - A decrease means the model predicts finance text more accurately.')

## Step 5: Apply QLoRA (The Fine-Tuning Setup)

### What is LoRA? (Very simple explanation)

Imagine you have a math textbook (TinyLlama). It's huge. You want to add extra notes about finance to it.

- **Normal fine-tuning** = Rewrite the entire textbook (expensive, slow, needs huge memory)
- **LoRA** = Just add sticky notes on top of relevant pages (cheap, fast, same quality)

Technically, LoRA adds two small matrices (A and B) to each attention layer.
Instead of updating all 1.1 billion parameters, we only train these small adapter matrices.

### Key numbers to understand
- TinyLlama has **1.1 billion** parameters total
- With LoRA rank=16, we only train **~8 million** parameters = less than 1%
- This is why it fits on a free GPU!

### What is `r` (rank)?
Rank controls how many parameters the adapters have. Higher rank = more learning capacity but more memory. Rank 16 is a good default.

In [ ]:
# =============================================================
# CELL 12: Apply QLoRA to the model
# =============================================================

print('Step 1: Preparing model for 4-bit training...')
# This enables gradient computation in the 4-bit model
# (Gradients are the signals used to update the model during training)
model = prepare_model_for_kbit_training(model)
model.enable_input_require_grads()  # Required when using gradient checkpointing
print('✅ Model prepared for k-bit training')
print()

print('Step 2: Configuring LoRA...')
lora_config = LoraConfig(
    r=16,              # Rank: size of the adapter matrices.
                       # Think of it as how many 'channels' of new information we add.
                       # r=16 is a good balance of quality vs speed.

    lora_alpha=32,     # Scaling factor = 2 * r (standard practice)
                       # Controls the magnitude of the LoRA updates

    target_modules=[   # Which weight matrices in the model to add adapters to
        'q_proj',      # Query projection (Attention: 'What am I looking for?')
        'k_proj',      # Key projection   (Attention: 'What does each word represent?')
        'v_proj',      # Value projection  (Attention: 'What info does each word carry?')
        'o_proj',      # Output projection (Combines all attention outputs)
    ],

    lora_dropout=0.05, # Randomly turn off 5% of adapter neurons during training
                       # This prevents the model from memorizing training data

    bias='none',       # Do not add bias terms to LoRA layers
    task_type=TaskType.CAUSAL_LM  # We are doing causal language modeling (text generation)
)

print('Step 3: Applying LoRA to the model...')
model = get_peft_model(model, lora_config)

print()
print('✅ LoRA applied!')
print()
print('Trainable parameter count:')
model.print_trainable_parameters()
print()
print('See that small percentage? That is the magic of LoRA.')
print('We only train a tiny fraction of the model, saving massive amounts of memory.')

## Step 6: Train!

This is the main event. Here's what happens during training:

1. **Forward pass**: The model sees a finance Q&A example and tries to predict the answer
2. **Loss calculation**: We measure how wrong the prediction was (this is the 'loss' number)
3. **Backward pass**: We calculate which LoRA parameters need to change, and by how much (this is backpropagation)
4. **Update**: We nudge those parameters in the right direction
5. **Repeat**: Do this for all 2,000 examples, 3 times (3 epochs)

### What numbers to watch:
- **`loss`**: Should start around 1.0-2.5 and go DOWN over time. If it's going down, learning is happening!
- **`eval_loss`**: Loss on the validation set. Should also decrease but slower.

In [ ]:
# =============================================================
# CELL 13: Train the model!
# =============================================================
# Expected time: 15-25 minutes on a free T4 GPU
# Watch the 'loss' value -- it should decrease over time.

training_args = TrainingArguments(
    output_dir='./checkpoints',            # Where to save model checkpoints
    num_train_epochs=3,                    # Go through all training data 3 times
    per_device_train_batch_size=4,         # Process 4 examples at once per GPU
    gradient_accumulation_steps=4,         # Wait 4 steps before updating weights
                                           # Effective batch size = 4 x 4 = 16
    gradient_checkpointing=True,           # Trade speed for memory (saves ~30% memory)
    warmup_steps=30,                       # Slowly increase learning rate for first 30 steps
                                           # (prevents large, unstable updates at the start)
    learning_rate=2e-4,                    # How big each update step is (0.0002)
                                           # Too high = unstable, too low = slow learning
    fp16=True,                             # Use 16-bit floating point for speed
    logging_steps=25,                      # Print training stats every 25 steps
    evaluation_strategy='epoch',           # Evaluate on validation set after each epoch
    save_strategy='epoch',                 # Save checkpoint after each epoch
    load_best_model_at_end=True,           # At the end, use the best checkpoint found
    report_to='none',                      # Don't send metrics to Weights & Biases etc.
    optim='paged_adamw_8bit',              # Memory-efficient optimizer (Adam variant)
)

# SFTTrainer = Supervised Fine-Tuning Trainer
# It handles all the training loop details for us
trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=train_data,
    eval_dataset=val_data,
    dataset_text_field='text',   # Which column contains the formatted text
    max_seq_length=512,          # Maximum token length per example
    tokenizer=tokenizer,
)

print('🚀 Starting training!')
print(f'Training on {len(train_data)} examples for 3 epochs.')
print(f'Validating on {len(val_data)} examples after each epoch.')
print()
print('Expected time: 15-25 minutes on T4 GPU.')
print('Watch the \'loss\' value -- lower is better!')
print('-' * 60)

trainer.train()

print()
print('✅ Training complete!')

In [ ]:
# =============================================================
# CELL 14: Save the fine-tuned model
# =============================================================
# IMPORTANT: Save before testing! Colab can disconnect at any time.

SAVE_PATH = './finance-tinyllama-qlora'

model.save_pretrained(SAVE_PATH)
tokenizer.save_pretrained(SAVE_PATH)

print(f'✅ Model saved to: {SAVE_PATH}')
print()
print('Files saved:')
print('  adapter_model.safetensors  -- the LoRA weights (only ~16-32 MB!)')
print('  adapter_config.json        -- LoRA configuration')
print('  tokenizer files            -- how to convert text <-> tokens')
print()
print('Important: We did NOT save the full TinyLlama weights (they are unchanged).')
print('The base model stays on HuggingFace. We only save the small adapter.')
print('This is another advantage of LoRA -- the artifact is tiny!')

## Step 7: Evaluate After Training

Now the moment of truth. We'll:
1. Ask the **same 5 questions** we asked before training
2. Calculate **perplexity** again on the same validation set
3. Put everything in a **comparison table**

Be honest with the results. Small-scale fine-tuning on 2,000 examples doesn't create a perfect model. But you should see:
- More finance-specific vocabulary and framing
- Lower perplexity on the validation set
- Answers that feel more 'at home' in finance domain

In [ ]:
# =============================================================
# CELL 15: Test model AFTER fine-tuning
# =============================================================
# Same questions as before. Let's see what changed.

print('=' * 65)
print('POST-TRAINING: Model responses AFTER fine-tuning')
print('=' * 65)
print()

after_responses = []
for i, question in enumerate(TEST_QUESTIONS, 1):
    print(f'Q{i}: {question}')
    print('-' * 50)
    response = generate_response(question)
    print(f'{response}')
    print()
    after_responses.append(response)

In [ ]:
# =============================================================
# CELL 16: Calculate post-training perplexity
# =============================================================

print('Calculating post-training perplexity...')
print()

after_perplexity = calculate_perplexity(val_data, sample_size=100)

# Calculate improvement percentage
improvement = ((baseline_perplexity - after_perplexity) / baseline_perplexity) * 100

print('=' * 50)
print('PERPLEXITY RESULTS (lower is better)')
print('=' * 50)
print(f'  Before fine-tuning : {baseline_perplexity:.2f}')
print(f'  After  fine-tuning : {after_perplexity:.2f}')
print(f'  Change             : {improvement:+.1f}%')
print('=' * 50)
print()
if improvement > 0:
    print(f'✅ Perplexity improved by {improvement:.1f}%')
    print('  The model is less confused by finance text after training.')
elif improvement > -5:
    print('⚠️  Perplexity roughly unchanged.')
    print('  This can happen with small datasets. Check qualitative responses above.')
else:
    print('❌  Perplexity got worse -- the model may be overfitting.')
    print('  Check: did the training loss actually decrease?')

In [ ]:
# =============================================================
# CELL 17: Final side-by-side comparison
# =============================================================
# This is the core 'result' of your project.
# Copy this output for your README.

print('=' * 70)
print('FULL BEFORE vs AFTER COMPARISON')
print('=' * 70)

for i, (question, before, after) in enumerate(
        zip(TEST_QUESTIONS, before_responses, after_responses), 1):

    print(f'\nQUESTION {i}: {question}')
    print()

    print('[ BEFORE fine-tuning ]')
    # Show up to 350 characters to keep output readable
    print(before[:350] + ('...' if len(before) > 350 else ''))
    print()

    print('[ AFTER fine-tuning ]')
    print(after[:350] + ('...' if len(after) > 350 else ''))
    print()
    print('-' * 70)

print()
print('QUANTITATIVE SUMMARY')
print('-' * 30)
print(f'Baseline perplexity    : {baseline_perplexity:.2f}')
print(f'Post-training perplexity: {after_perplexity:.2f}')
print(f'Improvement            : {improvement:+.1f}%')
print(f'Training samples used  : {len(train_data)}')
print(f'Epochs                 : 3')
print(f'LoRA rank              : 16')
print(f'Base model             : TinyLlama-1.1B-Chat')
print(f'Trainable params       : ~0.8% of total parameters')

## 🏁 You're Done!

### What you just did (for your interview)

> *"I fine-tuned TinyLlama-1.1B using QLoRA (4-bit NF4 quantization + LoRA rank-16 adapters targeting the Q/K/V/O projections) on 2,000 financial Q&A examples from the finance-alpaca dataset. Training ran for 3 epochs on a T4 GPU. I measured perplexity on a 200-sample held-out validation set before and after training, and qualitatively compared responses on 5 benchmark finance questions."*

### What you can defend in an interview
- Why QLoRA instead of full fine-tuning? (memory constraints, free GPU)
- Why LoRA rank 16? (standard starting point, good tradeoff)
- What does perplexity measure? (model's uncertainty per token, lower = better domain fit)
- What are the limitations? (small dataset, proxy metric, no external benchmark evaluation)

### Limitations (be honest about these)
- 2,000 training examples is small. A production system would use much more.
- Perplexity on the training domain is a proxy metric. True quality requires human evaluation or held-out benchmarks.
- The base model (TinyLlama-1.1B) is intentionally small. Larger models would produce better results.